# Setup

## Package Installs

In [0]:
%pip install catboost lightgbm optuna duckdb tabulate shap

## Imports

In [0]:
import numpy as np
import pandas as pd
import duckdb
from tabulate import tabulate

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_log_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

## Global Vairables

In [0]:
def check_environment() -> str:
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

In [0]:
TARGET: str = "is_fraud"
N_SPLITS: int = 5
SEED: int = 42
DRIVE_FOLDER_NAME: str = "Credit Card Fraud"
ENVIRONMENT: str = check_environment()

non_relevant_columns: list[str] = ['id', 'source']
eps: float = 1e-5

## Helper Functions

### Load Data

In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Split Data

In [0]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

In [0]:
train_df, test_df, df = load_data(ENVIRONMENT)

# EDA

In [0]:
len(df)

In [0]:
df.info()

In [0]:
df.isna().sum()

In [0]:
df.describe().T

In [0]:
df.describe(include='object').T

## Categorical features

In [0]:
for col in df.select_dtypes(include='object').columns :
    print(f"\n=================Column: {col} =================")
    print(df[col].unique())
    print("----------------")
    print(df[col].value_counts())
    print("----------------")
    print(df[col].value_counts(normalize=True))

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

target = 'is_fraud'  # change to your target column
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type',
            'day_of_week', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
            'ip_country_mismatch', 'billing_shipping_mismatch',
            'is_ai_generated_scam_attempt']

base_rate = df[target].mean()
ncols = 3
nrows = int(np.ceil(len(cat_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))

for ax, col in zip(axes.flat, cat_cols):
    g = (df.groupby(col, observed=True)[target]
           .agg(n='size', rate='mean')
           .sort_values('rate', ascending=False))
    se = np.sqrt(g['rate'] * (1 - g['rate']) / g['n'])   # binomial standard error
    x = np.arange(len(g))

    ax.bar(x, g['n'], color='lightgrey')
    ax.set_ylabel('count')
    ax.set_xticks(x, g.index.astype(str), rotation=45, ha='right')
    ax.set_title(col)

    ax2 = ax.twinx()
    ax2.errorbar(x, g['rate'], yerr=1.96 * se, fmt='o', color='crimson', capsize=3)
    ax2.axhline(base_rate, ls='--', color='crimson', alpha=0.5)
    ax2.set_ylabel('fraud rate')
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1))

for ax in axes.flat[len(cat_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [0]:
df.groupby(
    by="merchant_category"
)['is_fraud'].mean().sort_values(ascending=False).index

In [0]:
import matplotlib.ticker as mtick

cat_cols = df.select_dtypes(include=["object", "bool"]).columns
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))

fig, axs = plt.subplots(ncols=n_cols, nrows=n_rows, figsize=(6 * n_cols, 4 * n_rows))
for ax, col in zip(axs.flat, cat_cols):
    mean_order = (
        df.groupby(by=f"{col}")[TARGET]
        .mean()
        .sort_values(ascending=False)
        .index
    )

    sns.countplot(
        data=df, x=f"{col}", ax=ax, color="lightgrey", order=mean_order
    )

    ax_1 = ax.twinx()
    sns.pointplot(
        data=df,
        x=f"{col}",
        y=TARGET,
        color="crimson",
        ax=ax_1,
        err_kws={"linewidth": 1},
        linewidth=2,
        markersize=5,
    )

    ax_1.axhline(
        y=df[TARGET].mean(),
        ls="--",
        color="crimson",
        alpha=0.8,
        label="overall fraud rate",
    )
    ax_1.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"{col}")
    ax.tick_params(axis="x", rotation=45)

plt.legend()
plt.tight_layout()